In [3]:

from pathlib import Path,sys
sys.path.append( "../")
sys.path.append( "../../")

from typing import Any, Dict
import json
import yaml
from get_llm_model import *
import pandas as pd, numpy as np
import duckdb
import pprint 



inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year
print( inj.sample(3))


llm = azure_llm_if()
print( llm )



imported
          DATE NAME  WATER_INJECTION_VOLUME  SECTOR  ZONE SUBZONE WELL_TYPE  \
386 2023-08-02   I4                 1184.74       1  WARA   WARA1  Injector   
60  2020-12-02   I1                 1655.37       1  WARA   WARA1  Injector   
306 2016-12-02   I4                  591.30       1  WARA   WARA1  Injector   

     DAY  MONTH  YEAR  
386    2      8  2023  
60     2     12  2020  
306    2     12  2016  
client=<openai.resources.chat.completions.completions.Completions object at 0x750b81323490> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x750b7fa21990> root_client=<openai.lib.azure.AzureOpenAI object at 0x750b81323250> root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x750b80399330> model_name='gpt-4o' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True azure_endpoint='https://openai-if-test.openai.azure.com/' deployment_name='gpt-4' openai_api_version='2024-12-01-preview' opena

In [5]:
from typing import Dict, List, Tuple, Optional, Any 
def sanitize_df(df):

    df = df.copy()

    # Ensure index is not problematic
    if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
        df = df.reset_index()

    # Attempt to convert object columns
    for col in df.columns:
        if df[col].dtype == "object":
            # try datetime
            converted = pd.to_datetime(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted
                continue

            # try numeric
            converted = pd.to_numeric(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted

    return df

def register_table(conn, name: str, df, registry: set, columns: Dict[str, Any]):
    df = sanitize_df(df)
    conn.register(name, df)



In [6]:

con = duckdb.connect()
con.register("injectors", sanitize_df(inj))


/tmp/ipykernel_13282/3044984730.py:14: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_13282/3044984730.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(df[col], errors="ignore")
/tmp/ipykernel_13282/3044984730.py:20: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  converted = pd.to_numeric(df[col], errors="ignore")
/tmp/ipykernel_13282/3044984730.py:14: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  converted

In [7]:
from load_semantics import load_semantics
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )


In [8]:
semantic_catalog.tables[0]

TableCard(name='injectors', description='Water-injection time series per injector well, subzone and sector', columns=[ColumnCard(name='DATE', data_type='timestamp', description='Timestamp of injection observation.', business_rules=[]), ColumnCard(name='NAME', data_type='string', description='Injector well name. Unique identifier for the injector well', business_rules=[]), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume for the period.', business_rules=[]), ColumnCard(name='SUBZONE', data_type='string', description='Subzone name. A subzone indicates vertical interval', business_rules=['Expected values include labels such as LW, RW, UW, Unique.']), ColumnCard(name='SECTOR', data_type='integer', description='Sector identifier. A sector indicates a geographical location', business_rules=['Should be an integer sector id.']), ColumnCard(name='YEAR', data_type='integer', description='Year component of DATE.', business_rules=['If present, should 

In [9]:
idioms = {
    "duckdb": {
        "date_subtraction": "Use column - INTERVAL 'X days/months'. NEVER use DATE_SUB() or DATEADD().",
        "date_truncation": "Use DATE_TRUNC('month', column).",
        "reserved_keywords": 'Always wrap the column name "DATE" in double quotes to avoid Binder Errors.',
        "string_concatenation": 'Use the || operator or CONCAT().',
        "boolean_aggregation": 'Use FILTER clauses or BOOL_OR() / BOOL_AND() for cleaner logic.'
    }
}

prompt_template = """
You are an expert SQL generator for DuckDB based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

# Task: 
{user_query}

# DuckDB SQL:
"""



query = "Summarize the injection data as described in the business context."

idiom_name = "duckdb"
idiom_examples = "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])
#context = semantic_catalog.tables[0].model_dump_json()
# Instead of model_dump_json(), use:
#context = json.dumps(json.loads(semantic_catalog.tables[0].model_dump_json()), indent=2)

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)


# FIX: Changed 'idioms' to 'idiom' to match the template placeholder
prompt = prompt_template.format(
    user_query=query, 
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)


You are an expert SQL generator for DuckDB based on the 
following database schema and description:

# Tables:
name: injectors
description: Water-injection time series per injector well, subzone and sector
columns:
- name: DATE
  data_type: timestamp
  description: Timestamp of injection observation.
  business_rules: []
- name: NAME
  data_type: string
  description: Injector well name. Unique identifier for the injector well
  business_rules: []
- name: WATER_INJECTION_VOLUME
  data_type: float
  description: Injected water volume for the period.
  business_rules: []
- name: SUBZONE
  data_type: string
  description: Subzone name. A subzone indicates vertical interval
  business_rules:
  - Expected values include labels such as LW, RW, UW, Unique.
- name: SECTOR
  data_type: integer
  description: Sector identifier. A sector indicates a geographical location
  business_rules:
  - Should be an integer sector id.
- name: YEAR
  data_type: integer
  description: Year component of DATE.

In [10]:
query1 = "how many wells are there?"
query2 = "What is the total water injection volume by year?"
query3 = "Tell me the mean yearly water injection volume for each subzone"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"


queries = [
    (query1, lambda x: int(x.loc[0,:].values[0]) == 5 ),
    (query2, 
    lambda x: abs(float(x.loc[ result['YEAR'] == 2016, :].values[0][1]) - 56202.305) < 0.01),
    (query3, lambda x: False),
    (query4, lambda x: False),
    (query5, lambda x: False),
    (query6, lambda x: False),
    ]

for n,query_item in enumerate(queries):
    query = query_item[0]
    print(60 * '=')
    print(query)
   
    prompt = prompt_template.format(
    user_query=query, 
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
    ) 
    response = llm.invoke(prompt)

    print(response)

    # execute the generated SQL
    sql = response.content.strip()
    print(sql)

    try: 
        result = con.execute(sql).fetchdf()
        display(result.sample( min(3, result.shape[0]) ))
        
        # check result
        checking_fn = query_item[1]
        print('success', checking_fn(result))
    except Exception as e:
        print('error executing SQL:', e)





how many wells are there?
content='SELECT COUNT(DISTINCT NAME) AS well_count FROM injectors' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 671, 'total_tokens': 685, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DTpVSRJsHqHcv1cPP4yKM67EMK0ci', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'sev

,well_count
0,5


success True
What is the total water injection volume by year?
content='SELECT YEAR, SUM(WATER_INJECTION_VOLUME) AS total_water_injection_volume FROM injectors GROUP BY YEAR' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 675, 'total_tokens': 698, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DTpVScC3WEWZIEKgzZJp9Y8bJq6HE', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'st

,YEAR,total_water_injection_volume
3,2017,64580.163
6,2021,64727.558
0,2016,56202.305


success True
Tell me the mean yearly water injection volume for each subzone
content='SELECT SUBZONE, YEAR, AVG(WATER_INJECTION_VOLUME) AS MEAN_YEARLY_WATER_INJECTION_VOLUME \nFROM injectors \nGROUP BY SUBZONE, YEAR' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 678, 'total_tokens': 713, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DTpVTmudbU9JdZkDUyrO5FXTRQiy0', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 's

...

success False
whats the frequency of observations in the dataset (D, M, Y) ?
content="SELECT CASE \n    WHEN COUNT(DISTINCT YEAR) > 1 THEN 'Y'\n    WHEN COUNT(DISTINCT MONTH) > 1 THEN 'M'\n    ELSE 'D'\nEND AS observation_frequency\nFROM injectors" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 682, 'total_tokens': 729, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DTpVU7RJmSPeAfd9LYtKoaeqSPEaV', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'v

,observation_frequency
0,Y


success False
summarize the injection data
content='WITH current_date AS (\n    SELECT MAX("DATE") AS max_date FROM injectors\n),\nlast_three_months AS (\n    SELECT \n        DATE_TRUNC(\'month\', "DATE") AS month,\n        SUBZONE,\n        NAME,\n        SUM(WATER_INJECTION_VOLUME) AS total_injection_volume\n    FROM injectors\n    WHERE "DATE" >= (SELECT max_date - INTERVAL \'3 months\' FROM current_date)\n    GROUP BY DATE_TRUNC(\'month\', "DATE"), SUBZONE, NAME\n),\nactive_injectors AS (\n    SELECT DISTINCT NAME\n    FROM injectors\n    WHERE WATER_INJECTION_VOLUME > 0\n      AND "DATE" >= (SELECT max_date - INTERVAL \'3 months\' FROM current_date)\n),\ninactive_injectors AS (\n    SELECT DISTINCT NAME\n    FROM injectors\n    WHERE NAME NOT IN (SELECT NAME FROM active_injectors)\n),\nsummary AS (\n    SELECT \n        month,\n        SUBZONE,\n        COUNT(DISTINCT NAME) AS active_injectors_count,\n        SUM(total_injection_volume) AS total_injection_volume\n    FROM last_th

,month,SUBZONE,active_injectors_count,total_injection_volume,inactive_injectors_count
0,2023-11-01,WARA1,5,6583.0,0
2,2023-10-01,WARA1,5,6571.0,0
1,2023-12-01,WARA1,5,14523.0,0


success False


In [11]:
response.response_metadata['token_usage']['prompt_tokens']

672

In [31]:


class SmartData:
    def __init__(self, llm=None):
        self.con = duckdb.connect()
        self.tables = set()
        self.semantic = {}
        self.semantic_model = load_semantic_model()
        self.columns = {}
        self._llm = llm

    # -----------------------------
    # LLM PROPERTY
    # -----------------------------
    @property
    def llm(self):
        return self._llm

    @llm.setter
    def llm(self, model):
        self._llm = model

    # -----------------------------
    # CORE EXECUTION
    # -----------------------------
    def execute_query(self, sql: str):
        return self.con.execute(sql).fetchdf()

    # -----------------------------
    # TABLE REGISTRATION
    # -----------------------------
    def register_tables(self, tables: Dict[str, Any]):
        register_tables(self.con, tables, self.tables, self.columns)

    def register_table(self, name: str, df):
        register_table(self.con, name, df, self.tables, self.columns)

    # -----------------------------
    # SEMANTIC REGISTRATION
    # -----------------------------
    def register_semantic(self, name: str, description: str):
        if name not in self.tables:
            raise ValueError(f"Table '{name}' is not registered")
        self.semantic[name] = description

    # -----------------------------
    # SQL GENERATION
    # -----------------------------
    def generate_sql(self, user_query: str, **kwargs) -> str:
        if self._llm is None:
            raise ValueError("LLM is not set")

        context_lines = []
        for table in self.tables:
            desc = self.semantic.get(table, "")
            cols = self.columns.get(table, [])
            context_lines.append(
                f"Table: {table}\nDescription: {desc}\nColumns: {', '.join(cols)}"
            )

        context = "\n\n".join(context_lines)

        prompt = f"""
You are an expert SQL generator for DuckDB.

Available tables:
{context}

User request:
{user_query}

Generate a valid DuckDB SQL query only.
"""

        return self._llm(prompt, **kwargs)


llm = azure_llm_if()
print( llm )



client=<openai.resources.chat.completions.completions.Completions object at 0x0000018D5881CCA0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000018D588D72E0> root_client=<openai.lib.azure.AzureOpenAI object at 0x0000018D5881DA80> root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000018D5881E980> model_name='gpt-4o' temperature=0.0 model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True azure_endpoint='https://openai-if-test.openai.azure.com/' deployment_name='gpt-4' openai_api_version='2024-12-01-preview' openai_api_type='azure'
